# Field Transformations Quickstart

This notebook shows the exact workflow:
1. define fields,
2. build a model,
3. compile a source-basis Lagrangian,
4. apply `FieldTransformation` rules,
5. compare `.feynman_rule(...)` before vs after.


In [1]:
from __future__ import annotations
import re
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from symbolica import S

from feynpy import (
    CovD,
    Field,
    FieldTransformation,
    GaugeGroup,
    LORENTZ_INDEX,
    Model,
    WEAK_FUND_INDEX,
    replacement,
)


In [2]:
ANSI_ESCAPE_RE = re.compile(r"\x1B\[[0-?]*[ -/]*[@-~]")



def clean(text):
    return ANSI_ESCAPE_RE.sub("", str(text))

def show(title, result):
    print("==========")
    print(title)
    if isinstance(result, dict):
        print(f"{len(result)} vertex signature(s)")
        print()
        for signature, expression in result.items():
            print("Vertex:", signature)
            print("Rule:", clean(expression))
            print()
    else:
        print(clean(result))
        print()

## 1) Define fields and source-basis model

We use one charged scalar doublet `H` and one neutral gauge field `B`.
Then we define two physical vector fields `A` and `Z` that will be introduced
only through transformations.


In [3]:
mu = S("mu")
g1 = S("g1")
sw = S("sw")
cw = S("cw")
yH = S("yH")

B = Field(
    "B",
    spin=1,
    self_conjugate=True,
    symbol=S("B0"),
    indices=(LORENTZ_INDEX,),
)

A = Field(
    "A",
    spin=1,
    self_conjugate=True,
    symbol=S("A0"),
    indices=(LORENTZ_INDEX,),
)

Z = Field(
    "Z",
    spin=1,
    self_conjugate=True,
    symbol=S("Z0"),
    indices=(LORENTZ_INDEX,),
)

H = Field(
    "H",
    spin=0,
    self_conjugate=False,
    symbol=S("H0"),
    conjugate_symbol=S("Hdag0"),
    indices=(WEAK_FUND_INDEX,),
    quantum_numbers={"Y": yH},
)

U1Y = GaugeGroup(
    name="U1Y",
    abelian=True,
    coupling=g1,
    gauge_boson="B",
    charge="Y",
)

source_model = Model(
    name="transform-quickstart",
    gauge_groups=(U1Y,),
    fields=(H, B, A, Z),
    lagrangian_decl=CovD(H.bar, mu) * CovD(H, mu),
)


## 2) Before transformations

In the source basis, vertices exist with `B`, but not with `A`/`Z`.


In [4]:
def safe_rule(L, *fields):
    try:
        return L.feynman_rule(*fields)
    except ValueError:
        return 0


L_source = source_model.lagrangian()
show("l",L_source.feynman_rule())

print("================================================")
print("Source basis:")
print("  Γ(Hdag, H, B):", safe_rule(L_source, H.bar, H, B))
print("  Γ(Hdag, H, A):", safe_rule(L_source, H.bar, H, A))
print("  Γ(Hdag, H, Z):", safe_rule(L_source, H.bar, H, Z))


l
3 vertex signature(s)

Vertex: ('H.bar', 'H', 'B')
Rule: -1𝑖*g1*yH*g(cof(2, w1),cof(2, w2))*pcomp(q1,mu3)+1𝑖*g1*yH*g(cof(2, w1),cof(2, w2))*pcomp(q2,mu3)

Vertex: ('H.bar', 'H', 'B', 'B')
Rule: 2𝑖*g1^2*yH^2*g(mink(4, mu3),mink(4, mu4))*g(cof(2, w1),cof(2, w2))

Vertex: ('H.bar', 'H')
Rule: -1𝑖*g(cof(2, w1),cof(2, w2))*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)

Source basis:
  Γ(Hdag, H, B): -1𝑖*g1*yH*g(cof(2, w1),cof(2, w2))*pcomp(q1,mu3)+1𝑖*g1*yH*g(cof(2, w1),cof(2, w2))*pcomp(q2,mu3)
  Γ(Hdag, H, A): 0
  Γ(Hdag, H, Z): 0


## 3) Apply a field transformation

Now apply the electroweak-like linear rotation for `B`:
`B -> -sw * Z + cw * A`.

The replacement is written directly as a field expression, close to the
FeynRules `Definitions` style: `FieldTransformation(B, -sw * Z + cw * A)`.
Coefficients can be any parameters/expressions, sums become multiple terms,
and a bare scalar in the sum (e.g. a vev) becomes a constant shift. For
index-dependent or spinor-matrix rules (projectors, CKM) use `builder=`.


In [5]:
# FeynRules-like syntax: write the definition as a field expression.
# (Equivalent to terms=(replacement(-sw, Z), replacement(cw, A)).)
rules = (
    FieldTransformation(B, -sw * Z + cw * A),
)

L_physical = source_model.transform_fields(*rules, repeat=True)
show("l",L_physical.feynman_rule())

print("================================================")
print("After transformations:")
print("  Γ(Hdag, H, B):", safe_rule(L_physical, H.bar, H, B))
print("  Γ(Hdag, H, A):", safe_rule(L_physical, H.bar, H, A))
print("  Γ(Hdag, H, Z):", safe_rule(L_physical, H.bar, H, Z))


l
6 vertex signature(s)

Vertex: ('H.bar', 'H', 'Z')
Rule: 1𝑖*g1*sw*yH*g(cof(2, w1),cof(2, w2))*pcomp(q1,mu3)-1𝑖*g1*sw*yH*g(cof(2, w1),cof(2, w2))*pcomp(q2,mu3)

Vertex: ('H.bar', 'H', 'A')
Rule: -1𝑖*g1*cw*yH*g(cof(2, w1),cof(2, w2))*pcomp(q1,mu3)+1𝑖*g1*cw*yH*g(cof(2, w1),cof(2, w2))*pcomp(q2,mu3)

Vertex: ('H.bar', 'H', 'Z', 'Z')
Rule: 2𝑖*g1^2*sw^2*yH^2*g(mink(4, mu3),mink(4, mu4))*g(cof(2, w1),cof(2, w2))

Vertex: ('H.bar', 'H', 'Z', 'A')
Rule: -2𝑖*g1^2*sw*cw*yH^2*g(mink(4, mu3),mink(4, mu4))*g(cof(2, w1),cof(2, w2))

Vertex: ('H.bar', 'H', 'A', 'A')
Rule: 2𝑖*g1^2*cw^2*yH^2*g(mink(4, mu3),mink(4, mu4))*g(cof(2, w1),cof(2, w2))

Vertex: ('H.bar', 'H')
Rule: -1𝑖*g(cof(2, w1),cof(2, w2))*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)

After transformations:
  Γ(Hdag, H, B): 0
  Γ(Hdag, H, A): -1𝑖*g1*cw*yH*g(cof(2, w1),cof(2, w2))*pcomp(q1,mu3)+1𝑖*g1*cw*yH*g(cof(2, w1),cof(2, w2))*pcomp(q2,mu3)
  Γ(Hdag, H, Z): 1𝑖*g1*sw*yH*g(cof(2, w1),cof(2, w2))*pcomp(q1,mu3)-1𝑖*g1*sw*yH*g(cof(2, w1),cof(2, w2))*

## 4) What happened

- `B` was never a Python variable reassignment.
- The transformation rewrote each `B` occurrence inside compiled interaction terms.
- Therefore `.feynman_rule(...)` changes basis automatically:
  source-basis vertices disappear, physical-basis vertices appear.
